# TIL 2026-11-17 — Prompt & Context Engineering (Lesson 03)

## 오늘 한 일
- `prompt_compare.py`로 기본 프롬프트와 구조화 프롬프트를 Preview로 비교했다.
- `--context`로 참고 자료를 넣고 `<reference>`에 들어가는지 확인했다.
- `validate_format()`의 동작과 한계를 확인했다.
- 같은 질문을 기본/구조화 프롬프트로 Live 비교했다. (출력 상한 128, 결과 PARTIAL)

## 테스트 기록
| 항목 | 결과 |
|---|---|
| 기존 Lesson 01~02 회귀 테스트 | 성공 (전체 64개 `OK`에 포함) |
| Lesson 03 신규 테스트 | 성공, 9개 |
| Preview 모드 | 기본/구조화 프롬프트 출력, `[PREVIEW]` 문구 확인 |
| Context 삽입 | 입력한 공지 문장이 `<reference>` 안에 들어감 |
| Live API 요청 | 성공 (2회 호출), 답변은 둘 다 미완성 |
| 사용 모델 | `gpt-6-luna` |
| 응답 형식 | 구조화 응답에서 `핵심 설명`만 True (답이 잘려서) |

## Live 비교 결과 — PARTIAL
질문: `AI Agent와 챗봇의 차이는?` / 출력 상한: 128 (기본값, 변경 안 함)

| 평가 항목 | 기본 프롬프트 | 구조화 프롬프트 |
|---|---|---|
| 사용 모델 / 실행 여부 | `gpt-6-luna` / 실행 | `gpt-6-luna` / 실행 |
| 답변 이해도 | 판단 보류: 표 중간에서 답이 끊김 | 판단 보류: 첫 소제목 예시 중간에서 끊김 |
| 구체적인 예시 유무 | 표 형식 비교를 시작했으나 미완성 | 회의 일정 예시를 시작했으나 미완성 |
| 요청한 세 소제목 | 요구하지 않음 | `{'핵심 설명': True, '쉬운 예시': False, '확인할 점': False}` |
| 사실의 정확성 | 미검증 | 미검증 |
| 입력 토큰 수 | 17 | 160 |
| 출력 토큰 수 | 128 (상한 도달) | 128 (상한 도달) |
| 결론 및 한계 | 출력 상한에 걸려 미완성 | 출력 상한에 걸려 미완성. False는 형식 미준수가 아니라 잘림 때문 |

- 구조화 프롬프트는 입력 토큰이 약 9배 많았다. 역할·제약·형식 지시가 모두 입력 비용이 된다.
- 두 답변 모두 미완성이라 어느 쪽이 더 좋은지는 판단할 수 없다.
- 다시 비교하려면 같은 질문·같은 모델로 `AI_MAX_OUTPUT_TOKENS=512`처럼 상한을 올리고, 상한 값을 결과에 함께 기록한다.

## 배운 것
- **Prompt와 Context**: Prompt는 해야 할 일과 지시, Context는 참고 정보다. Context는 실제 요청 문자열에 들어가야만 모델이 쓸 수 있다.
- **프롬프트 다섯 요소**: 역할, 목표, 맥락, 제약, 출력 형식. 요소를 많이 쓴다고 정확해지지는 않는다.
- **역할과 Context 충돌**: `--context`에 "실무 개발자"라고 넣어도 고정 역할은 "입문자를 위한 학습 도우미"로 그대로다. 수준이 어긋난 프롬프트가 된다.
- **`validate_format()`의 한계**: 문자열 포함 여부만 본다. 제목 순서를 바꿔도, 중복해도, 문장 중간에 써도 모두 True다. 내용 정확성은 전혀 보지 않는다.
- **이 예제의 역할 구분은 보안 경계가 아니다**: 지시문을 하나의 사용자 입력 문자열로 합쳐 보낸다. 시스템/개발자 메시지를 쓰는 구조가 아니다.

## 확인 문제 — 내 답
1. Prompt는 요청과 지시, Context는 그 요청을 처리할 때 필요한 참고 정보다.
2. 역할은 답하는 관점과 난이도를 정하고, 참고 문서는 모델이 모르는 사실을 알려 준다.
3. 형식과 방향을 안내할 뿐 모델의 사실 확인 능력을 높여 주지는 않는다.
4. 형식 검사는 제목 문자열만 확인하고 내용은 확인하지 않는다.
5. 모델마다 토크나이저와 출력 경향이 달라서 다른 모델이면 토큰 수 차이가 프롬프트 때문인지 알 수 없다.

## Lesson 04 준비 — `document_qa.py` 점검
Lesson 04(RAG) 실습 파일을 수정한 뒤 실행이 되는지 확인했다. 결과는 **실행 불가**.

### 증상
- 문법 검사와 `tests.test_document_qa` 모두 1번째 줄에서 멈춤
- `SyntaxError: invalid character '“' (U+201C)`

### 원인: 코드를 복사해 붙여 넣으면서 형식이 깨짐
| 문제 | 원래 형태 |
|---|---|
| 둥근 따옴표 `“ ” ‘` | 곧은 따옴표 `" '` |
| `__future__`, `__file__`, `__name__`, `__main__`의 밑줄 `__`이 사라짐 | 밑줄 포함 |
| 함수·클래스 본문의 들여쓰기가 사라짐 | 4칸 들여쓰기 |

블로그나 문서 편집기는 따옴표를 자동으로 "예쁜 따옴표"로 바꾸고, Markdown은 `__텍스트__`를 굵은 글씨로 해석해 밑줄을 지운다. 코드는 코드 블록이나 원본 파일에서 복사해야 한다.

### 코드 내용의 변경 (형식 문제와 별개)
- `tokenize()`가 `set`(중복 제거) 대신 `list`(중복 유지)를 반환한다. 점수가 "질문 단어가 섹션에 나온 전체 횟수"로 바뀌었다.
- 점수 계산에 섹션 제목도 포함한다.
- 점수가 같으면 문서 순서를 유지한다.
- 기존 테스트가 `set` 방식을 전제로 하면, 문법을 고친 뒤에도 테스트가 실패할 수 있다.

### 다음에 할 일
- [ ] 형식만 복구(로직 유지)할지, `git restore`로 원본을 되돌릴지 결정
- [ ] 복구 후 `python -m unittest discover -s tests -v`로 전체 테스트 확인
- [ ] Lesson 04 Preview 실습 진행 (`data/lesson04_notice.md` 사용)
- [ ] 쌓인 변경(`main.py`, `model_info.py`, `.env.example`, `document_qa.py`, TIL 파일) 정리 후 커밋

## 오늘의 한 줄
지시를 많이 붙이면 입력 비용이 늘고, 출력 상한이 작으면 답이 잘린다. 결과는 "실행 여부"와 "완성 여부"를 나눠서 기록해야 한다.
